# Experiment results

Reads what `experiments/run.py` and `experiments/self_learning.py` wrote to `results/`. Re-run those scripts to refresh. Numbers are on the **test** set (thresholds tuned on validation).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from nids.experiments import FIGURE_ROWS, RESULTS_DIR

def comparison(scenario):
    tables = {p.stem.removesuffix('_comparison'): pd.read_csv(p, index_col=0)
              for p in sorted((RESULTS_DIR / scenario).glob('*_comparison.csv'))}
    acc = pd.DataFrame({name: t['accuracy'] for name, t in tables.items()}).loc[list(FIGURE_ROWS)]
    return acc.rename(index=FIGURE_ROWS).style.format('{:.2%}').highlight_max(color='#d4edda')

## Scenario 2 — CICIDS2017, homogeneous detectors

Paper (Section V-B): detector 1 proposed solution 88.36%; detector 2 equal to its best single classifier, 74.57%.

In [ ]:
comparison('scenario2')

In [ ]:
display(Image(RESULTS_DIR / 'scenario2' / 'comparison.png'))

## Scenario 1 — NSL-KDD, heterogeneous detectors

`kddtest` tests on KDDTest+ (unseen attack types); `holdout` on held-out KDDTrain+ records.

In [ ]:
comparison('scenario1_kddtest')

In [ ]:
comparison('scenario1_holdout')

## Self-learning

Each detector's **standalone** accuracy (its own verdict, before any advice) per chunk of the time-ordered stream, with and without retraining on advised samples.

In [ ]:
chunks = pd.read_csv(RESULTS_DIR / 'self_learning' / 'chunks.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, group) in zip(axes, chunks.groupby('detector')):
    for retrain, g in group.groupby('retrain'):
        ax.plot(g['chunk'], g['standalone_accuracy'], marker='o',
                label='self-learning' if retrain else 'frozen')
    ax.set(title=name, xlabel='chunk (time order)', ylabel='standalone accuracy')
    ax.legend()
plt.show()

In [ ]:
chunks[chunks['retrain']][['chunk', 'detector', 'labels', 'learned']].pivot(index=['chunk', 'labels'], columns='detector', values='learned')